In [ ]:
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

# ---------------- DOWNLOAD DATA ---------------- #
data = yf.download("RELIANCE.NS", start="2011-01-01", end="2024-01-01")
data.columns = data.columns.get_level_values(0)

sensex = yf.download("^BSESN", start="2011-01-01", end="2024-01-01")
sensex.columns = sensex.columns.get_level_values(0)

data = data.join(
    sensex[["Close"]].rename(columns={"Close": "Sensex_Close"}),
    how="inner"
)

# ---------------- FEATURE ENGINEERING ---------------- #
data["log_return"] = np.log(data["Close"] / data["Close"].shift(1))
data["open_close_diff"] = (data["Close"] - data["Open"]) / data["Open"]
data["close_pct_change"] = data["Close"].pct_change()

# Z-score
data["rolling_mean_30"] = data["Close"].rolling(30).mean()
data["rolling_std_30"] = data["Close"].rolling(30).std()
data["z_score_30"] = (
    data["Close"] - data["rolling_mean_30"]
) / data["rolling_std_30"]

# RSI
delta = data["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / avg_loss

data["RSI"] = 100 - (100 / (1 + rs))

# ATR
high_low = data["High"] - data["Low"]
high_close = np.abs(data["High"] - data["Close"].shift(1))
low_close = np.abs(data["Low"] - data["Close"].shift(1))

true_range = np.maximum(high_low, np.maximum(high_close, low_close))
data["ATR"] = true_range.rolling(14).mean()

# Bollinger Bands
data["BB_MA_20"] = data["Close"].rolling(20).mean()
data["BB_std_20"] = data["Close"].rolling(20).std()
data["BB_position"] = (data["Close"] - data["BB_MA_20"]) / data["BB_std_20"]

# MACD
data["EMA_12"] = data["Close"].ewm(span=12, adjust=False).mean()
data["EMA_26"] = data["Close"].ewm(span=26, adjust=False).mean()

data["MACD"] = data["EMA_12"] - data["EMA_26"]
data["MACD_signal"] = data["MACD"].ewm(span=9, adjust=False).mean()
data["MACD_diff"] = data["MACD"] - data["MACD_signal"]

# Momentum + Volume
data["mom_5"] = data["Close"].pct_change(5)
data["volume_change"] = data["Volume"].pct_change()
data["volume_ma20"] = data["Volume"].rolling(20).mean()
data["volume_ratio"] = data["Volume"] / data["volume_ma20"]

# Sensex features
data["sensex_return"] = data["Sensex_Close"].pct_change()
data["rel_return"] = data["close_pct_change"] - data["sensex_return"]


# Target
data["target"] = (data["Close"].shift(-5) > data["Close"]).astype(int)

# ---------------- CLEAN DATA ---------------- #
data = data.drop(columns=[
    "rolling_mean_30", "rolling_std_30",
    "EMA_12", "EMA_26",
    "BB_MA_20", "BB_std_20",
    "MACD_signal", "MACD",
    "volume_ma20",
    "Sensex_Close",
])

data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna()
data = data[:-5]

# ---------------- SPLIT ---------------- #
split_date = "2021-01-01"

train = data[data.index < split_date]
test = data[data.index >= split_date]

features = [
    "log_return", "open_close_diff", "z_score_30",
    "RSI", "ATR", "BB_position", "MACD_diff", "mom_5",
    "volume_change", "volume_ratio",
    "rel_return"
]

X_train = train[features]
y_train = train["target"]

X_test = test[features]
y_test = test["target"]

# ---------------- SCALING ---------------- #
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------- SAMPLE WEIGHTS ---------------- #
time_weights = np.exp(np.linspace(0, 0.5, len(X_train)))

train_z = train.loc[X_train.index, "z_score_30"]
latest_z = train_z.iloc[-1]
z_diff = np.abs(train_z - latest_z)

regime_weights = 1 / (1 + z_diff)

weights = time_weights * regime_weights
weights = weights / np.mean(weights)

# ---------------- GRID SEARCH ---------------- #
tscv = TimeSeriesSplit(n_splits=5)

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    {"C": [0.01, 0.1, 1, 10]},
    cv=tscv,
    scoring="f1"
)
lr_grid.fit(X_train_scaled, y_train)

xgb_grid = GridSearchCV(
    XGBClassifier(eval_metric="logloss"),
    {
        "n_estimators": [50, 100],
        "max_depth": [3, 5],
        "learning_rate": [0.05, 0.1]
    },
    cv=tscv,
    scoring="f1"
)
xgb_grid.fit(X_train, y_train)

# ---------------- MODELS ---------------- #
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train, sample_weight=weights)

dt_model = DecisionTreeClassifier(max_depth=5)
dt_model.fit(X_train, y_train, sample_weight=weights)

ada_model = AdaBoostClassifier(n_estimators=100)
ada_model.fit(X_train, y_train)

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    eval_metric="logloss"
)
xgb_model.fit(X_train, y_train, sample_weight=weights)

lgb_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8
)
lgb_model.fit(X_train, y_train)

cat_model = CatBoostClassifier(
    iterations=200,
    depth=5,
    learning_rate=0.05,
    verbose=0
)
cat_model.fit(X_train, y_train)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)
rf_model.fit(X_train, y_train)

# ---------------- PREDICTIONS ---------------- #
lr_pred = lr_model.predict(X_test_scaled)
dt_pred = dt_model.predict(X_test)
ada_pred = ada_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)
lgb_pred = lgb_model.predict(X_test)
cat_pred = cat_model.predict(X_test)
rf_pred = rf_model.predict(X_test)

# ---------------- ML METRICS ---------------- #
print("\nML PERFORMANCE\n")

print("LR F1:", f1_score(y_test, lr_pred))
print("DT F1:", f1_score(y_test, dt_pred))
print("ADA F1:", f1_score(y_test, ada_pred))
print("XGB F1:", f1_score(y_test, xgb_pred))
print("LGBM F1:", f1_score(y_test, lgb_pred))
print("CatBoost F1:", f1_score(y_test, cat_pred))
print("Random Forest F1:", f1_score(y_test, rf_pred))

print("\nConfusion Matrix (XGB):")
print(confusion_matrix(y_test, xgb_pred))

# ---------------- TRADING ---------------- #
def simulate_trading(predictions, test_df):
    capital = 1_000_000
    returns = []
    wins = 0
    trades = 0

    test_df = test_df.reset_index(drop=True)
    i = 0

    while i < len(test_df) - 6:
        signal = predictions[i]

        open_price = test_df.loc[i+1, "Open"]
        close_price = test_df.loc[i+5, "Close"]

        if signal == 1:
            trade_return = (close_price / open_price) - 1
            if close_price > open_price:
                wins += 1
        else:
            trade_return = (open_price / close_price) - 1
            if close_price < open_price:
                wins += 1

        capital *= (1 + trade_return)
        returns.append(trade_return)

        trades += 1
        i += 5

    return capital, np.array(returns), trades, wins


def actual_trading(test_df):
    capital = 1_000_000
    test_df = test_df.reset_index(drop=True)

    i = 0
    while i < len(test_df) - 6:
        open_price = test_df.loc[i+1, "Open"]
        close_price = test_df.loc[i+5, "Close"]

        if close_price > test_df.loc[i, "Close"]:
            trade_return = (close_price / open_price) - 1
        else:
            trade_return = (open_price / close_price) - 1

        capital *= (1 + trade_return)
        i += 5

    return capital

# ---------------- RUN BACKTEST ---------------- #
models = {
    "Logistic Regression": lr_pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred,
    "AdaBoost": ada_pred,
    "XGBoost": xgb_pred,
    "LightGBM": lgb_pred,
    "CatBoost": cat_pred
}

print("\n================ FINANCIAL PERFORMANCE ================\n")

for name, preds in models.items():
    cap, returns, trades, wins = simulate_trading(preds, test)

    total_return = (cap / 1_000_000 - 1) * 100

    if len(returns) > 1:
        sharpe = (np.mean(returns) - (0.05 / 252)) / np.std(returns)
        sharpe *= np.sqrt(252)
    else:
        sharpe = 0

    win_rate = (wins / trades) * 100 if trades > 0 else 0

    print(f"--- {name} ---")
    print(f"Final Capital: ₹{cap:.2f}")
    print(f"Total Return: {total_return:.2f}%")
    print(f"Sharpe Ratio: {sharpe:.3f}")
    print(f"Trades: {trades}")
    print(f"Win Rate: {win_rate:.2f}%\n")

# ---------------- ACTUAL BENCHMARK ---------------- #
actual_capital = actual_trading(test)

print("==================================================")
print(f"\nActual Strategy Capital (Perfect Foresight): ₹{actual_capital:.2f}")

# ---------------- FEATURE IMPORTANCE ---------------- #
importance = xgb_grid.best_estimator_.feature_importances_

plt.barh(features, importance)
plt.title("Feature Importance (XGBoost)")
plt.show()